# Train wav2vec2 Quran ASR + corrector — Colab (Option A, no HF Hub)

**Storage:** audio + dataset + caches on Colab `/content` (~70 GB, ephemeral); trained **checkpoints** on Google Drive.
**No Hugging Face account / Hub:** the trained model is saved to Drive only (not pushed anywhere).
Switch runtime to **GPU (T4)** before running.


## 0. Route HF *library* caches to the big /content partition (FIRST)

Not using the HF Hub as a service — this only caches the one-time base-model download (~1.2 GB)
and the `input_values` cache (~10 GB) on the big `/content` disk instead of the small ~12 GB root.


In [ ]:
import os
os.environ['HF_HOME'] = '/content/hf_cache'
os.makedirs('/content/hf_cache', exist_ok=True)

!nvidia-smi
!df -h /content | tail -1


## 1. Install dependencies


In [ ]:
!pip install -q -U pip
!pip install -q torch torchaudio transformers datasets accelerate evaluate jiwer \
    librosa soundfile huggingface_hub pyyaml requests tqdm
!apt-get -qq install -y ffmpeg > /dev/null

## 2. Get the code
Clone from your GitHub fork, or upload the project folder.


In [ ]:
# !git clone https://github.com/<you>/model-asr-quran.git /content/model-asr-quran
%cd /content/model-asr-quran
!pip install -q -e .
!python scripts/cloud_preflight.py


## 3. Mount Google Drive (checkpoints + audio backup)

Drive holds the trained checkpoints **and** a backup of the audio (so reruns never re-download). Audio itself works off `/content`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/quran_asr/checkpoints', exist_ok=True)

# If Drive is FULL from a previous run, free it:
# !rm -rf /content/drive/MyDrive/quran_asr/data /content/drive/MyDrive/quran_asr/processed


## 4. Audio — smart provisioning (NO re-download on reruns)

This one cell does the right thing every session:
- audio already on `/content` → skip
- else a Drive backup exists → **restore** it (~3-5 min)
- else → **download once** (MP3, ~15-20 min) **and back it up to Drive**

So after the first download, errors/disconnects never cost a re-download — just re-run this cell. MP3 is used (~half the size of WAV, decoded on-the-fly by `soundfile`).

In [ ]:
import os, subprocess
def sh(cmd): subprocess.run(cmd, shell=True, check=False)

DRIVE = '/content/drive/MyDrive'
assert os.path.exists(DRIVE), 'Mount Google Drive first (cell 3).'
AUDIO = '/content/quran_asr/data/raw/audio'
BACKUP = DRIVE + '/quran_asr/audio.tar'   # Drive backup (MP3 ~3-4 GB, or WAV ~7-11 GB)

has_audio = os.path.isdir(AUDIO) and any(os.scandir(AUDIO))
has_backup = os.path.exists(BACKUP)

def backup():
    print('backing up audio to Drive (so reruns never re-download)...')
    sh('tar cf /content/audio.tar -C /content/quran_asr/data/raw audio')
    sh(f'cp /content/audio.tar "{BACKUP}"')
    sh('rm /content/audio.tar')
    print('backed up')

if has_audio and has_backup:
    print('audio + backup both present — nothing to do')
elif has_audio and not has_backup:
    backup()                                   # e.g. WAV from a previous run -> back it up
elif not has_audio and has_backup:
    print('restoring audio from Drive backup (~3-5 min)...')
    sh(f'cp "{BACKUP}" /content/audio.tar')
    sh('tar xf /content/audio.tar -C /content/quran_asr/data/raw')
    sh('rm /content/audio.tar')
    print('restored')
else:
    print('first run: downloading MP3 (~15-20 min, once)...')
    sh('python scripts/download.py --config configs/cloud_local.yaml --no-convert --max-workers 16 --qps 30')
    backup()

## 5. Build dataset + vocab


In [ ]:
!python scripts/build.py --config configs/cloud_local.yaml
!python scripts/build_vocab.py --config configs/cloud_local.yaml


## 6. Train
Checkpoints save to **Drive** (`logging.output_dir`). No HF Hub push.


In [ ]:
from quran_asr.config import Config
from quran_asr.training.train import train

cfg = Config.from_yaml('configs/cloud_local.yaml')   # logging.hub_repo stays null = no push
trainer, processor = train(cfg)
print('training done')


## 7. Evaluate


In [ ]:
!python scripts/evaluate.py --config configs/cloud_local.yaml --split test \
    --out /content/quran_asr/metrics.json


## 8. Corrector demo


In [ ]:
import glob
audio_file = glob.glob('/content/quran_asr/data/raw/audio/Husary_128kbps_Mujawwad/001001.*')[0]
!python scripts/correct.py --model-dir /content/drive/MyDrive/quran_asr/checkpoints \
    --audio "$audio_file" --text "بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ"

## Resume after an error / disconnect
1. Re-run cells 0→3 (env, install, code, mount Drive).
2. Re-run the **Audio** cell (4) — it **restores from the Drive backup (~3-5 min)**, NOT a re-download.
3. Re-run Build + Vocab (5), then Train (6).

Training resumes from the Drive checkpoint if you pass `resume=True` to `train()`.

## Getting your model out
The trained model + processor are in `/content/drive/MyDrive/quran_asr/checkpoints/final/`. Copy that folder to your backend machine to serve it.

## Out of disk on /content?
The audio backup is MP3 (~3-4 GB). If `/content` is tight, the input_values cache (~10-20 GB, on `/content`) is the other consumer — it is recomputed each session (~10-15 min) and not backed up (too big for Drive).